# Federated Training — Mammography
**AMS26-15 PMF** — Binary lesion detection · Flower simulation

**Prerequisites:** run `colab_data_preparation.ipynb` first to prepare and save client data to Google Drive.

This notebook:
1. Installs training dependencies
2. Mounts Google Drive and copies client data to `/content` for fast I/O
3. Clones the project repository
4. Configures `config.yaml` with the correct client paths
5. Runs federated simulation (Flower)
6. Plots training curves and saves results back to Drive


## 1 · Install dependencies

In [ ]:
!pip install -q flwr[simulation] torch torchvision pyyaml
print('Dependencies installed.')

## 2 · Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_DATA = '/content/drive/MyDrive/federated_mammography_data'
print('Drive mounted. Data directory:', DRIVE_DATA)

## 3 · Copy client data to /content
Copying from Drive to local storage avoids slow network reads during training.  
Skip this cell if you want to read directly from Drive (slower).

In [ ]:
import shutil

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

clients_on_drive = [
    'client1_inbreast',
    'client2_cbisddsm',
    'client3_mias',
]

for name in clients_on_drive:
    src = os.path.join(DRIVE_DATA, name)
    dst = os.path.join(DATA_DIR, name)
    if os.path.exists(dst):
        print(f'  {name} already in /content, skipping copy.')
    else:
        print(f'  Copying {name} ...')
        shutil.copytree(src, dst)
        print(f'  Done -> {dst}')

CLIENT1 = os.path.join(DATA_DIR, 'client1_inbreast')
CLIENT2 = os.path.join(DATA_DIR, 'client2_cbisddsm')
CLIENT3 = os.path.join(DATA_DIR, 'client3_mias')

for name, path in [('INbreast', CLIENT1), ('CBIS-DDSM', CLIENT2), ('MIAS', CLIENT3)]:
    ann  = os.path.join(path, 'annotations.csv')
    imgs = os.path.join(path, 'images')
    n_imgs = len(os.listdir(imgs)) if os.path.isdir(imgs) else 0
    print(f'  {name}: annotations={os.path.isfile(ann)}, images={n_imgs}')

## 4 · Clone project repository

In [ ]:
import sys

REPO_URL = 'https://github.com/MarijaGijic/Federated_project.git'
REPO_DIR = '/content/federated_project'

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print('Repo updated.')

PROJECT_ROOT = os.path.join(REPO_DIR, 'Federated_project')
sys.path.insert(0, PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)

## 5 · Configure training

In [ ]:
import yaml

config_path = os.path.join(PROJECT_ROOT, 'config.yaml')
with open(config_path) as f:
    cfg = yaml.safe_load(f)

cfg['data']['client_dirs'] = [CLIENT1, CLIENT2, CLIENT3]

# ── Tune these to your needs ─────────────────────────────────────────────────
cfg['data']['image_size']        = 256   # 256 fits in Colab RAM; use 512 if you have A100
cfg['model']['features']         = [32, 64, 128, 256]   # lighter UNet
cfg['federated']['num_rounds']   = 10
cfg['training']['local_epochs']  = 3
cfg['training']['batch_size']    = 2    # keep low to avoid OOM
cfg['training']['learning_rate'] = 0.001
# ─────────────────────────────────────────────────────────────────────────────

with open(config_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('config.yaml saved:')
print(f"  clients      : {len(cfg['data']['client_dirs'])}")
print(f"  image_size   : {cfg['data']['image_size']}")
print(f"  features     : {cfg['model']['features']}")
print(f"  rounds       : {cfg['federated']['num_rounds']}")
print(f"  local_epochs : {cfg['training']['local_epochs']}")
print(f"  batch_size   : {cfg['training']['batch_size']}")

## 6 · Run federated simulation

In [ ]:
!python {PROJECT_ROOT}/train_federated.py \
    --config {config_path}

## 7 · Training curves

In [ ]:
import json
import matplotlib.pyplot as plt

results_dir  = os.path.join(PROJECT_ROOT, 'results')
history_path = os.path.join(results_dir, 'history.json')

with open(history_path) as f:
    history = json.load(f)

rounds = [r for r, _ in history['losses_distributed']]
losses = [v for _, v in history['losses_distributed']]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(rounds, losses, marker='o', color='steelblue')
axes[0].set_title('Distributed loss per round')
axes[0].set_xlabel('Round')
axes[0].set_ylabel('Loss')
axes[0].grid(True)

if history.get('metrics_distributed'):
    for metric_name, values in history['metrics_distributed'].items():
        r_vals = [r for r, _ in values]
        m_vals = [v for _, v in values]
        axes[1].plot(r_vals, m_vals, marker='o', label=metric_name)
    axes[1].set_title('Metrics per round')
    axes[1].set_xlabel('Round')
    axes[1].legend()
    axes[1].grid(True)
else:
    axes[1].set_visible(False)

plt.tight_layout()
chart_path = os.path.join(results_dir, 'training_curves.png')
plt.savefig(chart_path, dpi=120)
plt.show()
print('Chart saved ->', chart_path)

## 8 · Save results to Google Drive

In [ ]:
DRIVE_RESULTS = os.path.join(DRIVE_DATA, 'results')
if os.path.exists(DRIVE_RESULTS):
    shutil.rmtree(DRIVE_RESULTS)
shutil.copytree(results_dir, DRIVE_RESULTS)
print('Results saved to Google Drive ->', DRIVE_RESULTS)

ckpt_dir = os.path.join(DRIVE_RESULTS, 'checkpoints')
if os.path.isdir(ckpt_dir):
    for ckpt in sorted(os.listdir(ckpt_dir)):
        print(' ', ckpt)